# 06 — Interpretabilidad SHAP

**TFM RunnAing** | Fase 6.1

- SHAP (TreeExplainer) sobre el modelo ganador (Lundberg & Lee, 2017)
- Globales: summary beeswarm + bar importance
- Locales (3-5 casos): waterfall + force plot

**Outputs**: `reports/figures/shap_*.png` + `reports/evaluation/shap_importance.csv`

In [ ]:
import sys
sys.path.insert(0, '..')

import pickle
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from pathlib import Path

from src.splits import group_train_val_test_split

np.random.seed(42)
MODELS_DIR = Path('../models')
FIG_DIR = Path('../reports/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)
EVAL_DIR = Path('../reports/evaluation')
EVAL_DIR.mkdir(parents=True, exist_ok=True)

shap.initjs()
print('SHAP:', shap.__version__)

## 1. Cargar modelo ganador y test set

In [ ]:
with open(MODELS_DIR / 'split_meta.json') as f:
    meta = json.load(f)
best_name = meta['best_model_name']
FEATURE_COLS = meta['feature_cols']
print('Modelo:', best_name)

with open(MODELS_DIR / 'best_model.pkl', 'rb') as f:
    model = pickle.load(f)

df = pd.read_parquet('../data/processed/sessions_features.parquet')
X = df[FEATURE_COLS]; y = df['trimp']; groups = df['userId']

_, _, X_test, _, _, y_test, _, _, _ = \
    group_train_val_test_split(X, y, groups, train_frac=0.70, val_frac=0.15, seed=42)
print(f'Test: {len(X_test):,} sesiones')

## 2. Cálculo de valores SHAP

In [ ]:
if best_name == 'LinearRegression':
    scaler = model.named_steps['scaler']
    X_sc = pd.DataFrame(scaler.transform(X_test), columns=FEATURE_COLS, index=X_test.index)
    explainer = shap.LinearExplainer(model.named_steps['model'], X_sc)
    shap_values = explainer.shap_values(X_sc)
    X_shap = X_sc
else:
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test)
    X_shap = X_test.copy()

print('SHAP values shape:', np.array(shap_values).shape)

## 3. Visualizaciones globales

In [ ]:
# Summary beeswarm
plt.figure(figsize=(9, 6))
shap.summary_plot(shap_values, X_shap, plot_type='dot', max_display=13, show=False)
plt.title(f'SHAP Summary (beeswarm) — {best_name}', fontsize=12, pad=10)
plt.tight_layout()
plt.savefig(FIG_DIR / 'shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: reports/figures/shap_beeswarm.png')

In [ ]:
# Bar importance
plt.figure(figsize=(8, 5))
shap.summary_plot(shap_values, X_shap, plot_type='bar', max_display=13, show=False)
plt.title(f'SHAP Feature Importance (mean |SHAP|) — {best_name}', fontsize=12, pad=10)
plt.tight_layout()
plt.savefig(FIG_DIR / 'shap_bar_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: reports/figures/shap_bar_importance.png')

In [ ]:
mean_abs = np.abs(shap_values).mean(axis=0)
shap_imp = pd.DataFrame({'feature': FEATURE_COLS, 'mean_abs_shap': mean_abs})
shap_imp = shap_imp.sort_values('mean_abs_shap', ascending=False)
print('Top-5 features:')
print(shap_imp.head(5).to_string(index=False))
shap_imp.to_csv(EVAL_DIR / 'shap_importance.csv', index=False)
print('Guardado: reports/evaluation/shap_importance.csv')

## 4. Visualizaciones locales (5 casos)

Casos: mediana TRIMP · P90 · P10 · mayor error · menor error

In [ ]:
y_arr = y_test.values
preds = model.predict(X_test)
errors = np.abs(y_arr - preds)

cases = {
    'mediana_trimp': int(np.argmin(np.abs(y_arr - np.median(y_arr)))),
    'trimp_alto_p90': int(np.argmin(np.abs(y_arr - np.percentile(y_arr, 90)))),
    'trimp_bajo_p10': int(np.argmin(np.abs(y_arr - np.percentile(y_arr, 10)))),
    'mayor_error': int(np.argmax(errors)),
    'menor_error': int(np.argmin(errors)),
}
for label, idx in cases.items():
    print(f'{label:<20}: idx={idx}, real={y_arr[idx]:.1f}, pred={preds[idx]:.1f}')

In [ ]:
if best_name != 'LinearRegression':
    explanation = explainer(X_shap)
    for label, idx in cases.items():
        plt.figure(figsize=(10, 5))
        shap.plots.waterfall(explanation[idx], max_display=10, show=False)
        plt.title(f'Waterfall — {label} | real={y_arr[idx]:.1f}, pred={preds[idx]:.1f}',
                  fontsize=11)
        fname = f'shap_waterfall_{label}.png'
        plt.savefig(FIG_DIR / fname, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'Guardado: reports/figures/{fname}')

In [ ]:
if best_name != 'LinearRegression':
    for label, idx in cases.items():
        shap.force_plot(
            explainer.expected_value, shap_values[idx],
            X_shap.iloc[idx], feature_names=FEATURE_COLS,
            matplotlib=True, show=False,
        )
        fname = f'shap_force_{label}.png'
        plt.savefig(FIG_DIR / fname, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'Guardado: reports/figures/{fname}')